# ARC-AGI-3 Component-Skill Prefill + Competition Scored Run v3

This notebook builds skills from all attached `/kaggle/input` resources, exports `agent/my_agent.py`, and uses the official ARC-AGI-3 competition gateway/framework path to produce the scored `submission.parquet`.

In [ ]:

from pathlib import Path
import ast
import csv
import hashlib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
import textwrap
import time
import zipfile
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

SEED = int(os.environ.get("NINE_ARC_SEED", "918"))
random.seed(SEED)

KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
if not WORK_DIR.exists():
    WORK_DIR = Path("/tmp/kaggle_working_fallback")
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
SKILL_DIR = DATA_DIR / "skills"
REPORT_DIR = DATA_DIR / "reports"
REPLAY_DIR = DATA_DIR / "replay"
AGENT_DIR = WORK_DIR / "agent"
for p in [DATA_DIR, SKILL_DIR, REPORT_DIR, REPLAY_DIR, AGENT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "seed": SEED,
    "input_root": str(KAGGLE_INPUT),
    "work_dir": str(WORK_DIR),
    "max_scan_mb_per_file": float(os.environ.get("NINE_ARC_SCAN_MAX_MB", "32")),
    "run_scorecard": os.environ.get("NINE_ARC_RUN_SCORECARD", "1"),
    "max_actions": int(os.environ.get("NINE_ARC_MAX_ACTIONS", os.environ.get("NINE_ARC_MAX_STEPS", "240"))),
    "source_url": os.environ.get("NINE_ARC_SOURCE_URL", "https://github.com/engine/arc-agi3-component-skill-competition-v3"),
}
print(json.dumps(CONFIG, indent=2))
print("KAGGLE_INPUT exists:", KAGGLE_INPUT.exists())
print("KAGGLE_IS_COMPETITION_RERUN:", bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN")))


In [ ]:

# Official competition runtime resolver.
# In Kaggle, use the offline wheel path. No internet is required or used.
import sys, subprocess, os, json, zipfile
from pathlib import Path

resolver_report = {
    "offline_wheel_dirs_checked": [],
    "local_paths_added": [],
    "pip_commands": [],
    "arc_import_before": False,
    "arc_import_after": False,
    "errors": [],
}

def can_import_arc() -> bool:
    try:
        import arc_agi  # noqa: F401
        import arcengine  # noqa: F401
        return True
    except Exception as e:
        resolver_report["last_import_error"] = repr(e)
        return False

resolver_report["arc_import_before"] = can_import_arc()

# First: official competition wheel directory.
official_wheel_dirs = [
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"),
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/wheels"),
]
for wheel_dir in official_wheel_dirs:
    resolver_report["offline_wheel_dirs_checked"].append(str(wheel_dir))
    if wheel_dir.exists():
        cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wheel_dir), "arc-agi", "python-dotenv"]
        resolver_report["pip_commands"].append(" ".join(cmd))
        try:
            subprocess.run(cmd, check=True)
            break
        except Exception as e:
            resolver_report["errors"].append(f"official wheel install failed from {wheel_dir}: {e!r}")

# Second: search loaded Kaggle inputs for wheels/path packages if official path is absent.
if not can_import_arc() and KAGGLE_INPUT.exists():
    candidates = []
    for p in KAGGLE_INPUT.rglob("*"):
        name = p.name.lower()
        if p.is_dir() and name in {"site-packages", "dist-packages", "src"}:
            candidates.append(p)
        elif p.is_dir() and (p / "arc_agi").exists():
            candidates.append(p)
        elif p.is_file() and p.suffix.lower() == ".whl" and ("arc_agi" in name or "arc-agi" in name or "arcagi" in name):
            try:
                cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(p)]
                resolver_report["pip_commands"].append(" ".join(cmd))
                subprocess.run(cmd, check=False)
            except Exception as e:
                resolver_report["errors"].append(f"wheel install failed {p}: {e!r}")
    for c in candidates[:80]:
        sys.path.insert(0, str(c))
        resolver_report["local_paths_added"].append(str(c))

resolver_report["arc_import_after"] = can_import_arc()
(REPORT_DIR / "arc_runtime_resolver.json").write_text(json.dumps(resolver_report, indent=2))
print(json.dumps(resolver_report, indent=2)[:4000])


In [ ]:

# Component + dataset discovery. Static only: never executes public notebooks/scripts.
MAX_BYTES = int(CONFIG["max_scan_mb_per_file"] * 1024 * 1024)
TEXT_EXTS = {".py", ".ipynb", ".md", ".txt", ".json", ".jsonl", ".csv", ".tsv", ".yaml", ".yml"}
DATA_EXTS = {".json", ".jsonl", ".csv", ".tsv", ".parquet"}
ARCHIVE_EXTS = {".zip", ".whl"}

COMPONENT_PATTERNS = {
    "agent_contract": ["class MyAgent", "choose_action", "is_done", "Agent("],
    "action_api": ["GameAction", "ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "set_data", "reasoning"],
    "click_targeting": ["x", "y", "coordinate", "centroid", "click", "ACTION6", "bbox"],
    "object_extraction": ["connected", "component", "bbox", "object", "nonzero", "color", "segmentation", "centroid"],
    "state_graph": ["state_hash", "visited", "transition", "graph", "BFS", "DFS", "frontier", "untried"],
    "replay_memory": ["replay", "experience", "transition", "jsonl", "grid_before", "grid_after", "changed"],
    "world_model": ["world model", "simulate", "predict", "verifier", "effect", "delta", "model"],
    "stagnation_recovery": ["stuck", "stagnant", "reset", "same_state", "no_change", "GAME_OVER"],
    "training_loop": ["scorecard", "arc.make", "get_environments", "env.step", "main.py --agent"],
    "skill_library": ["SkillLibrary", "Skill", "success_rate", "confidence", "trigger_conditions"],
}
ACTION_REGEX = re.compile(r"\b(?:GameAction\.)?(ACTION[1-6]|RESET)\b")
COORD_REGEX = re.compile(r"['\"]x['\"]\s*:\s*(\d{1,2}).{0,24}['\"]y['\"]\s*:\s*(\d{1,2})", re.S)

inventory = []
component_hits = []
action_counts = Counter()
by_game_action_counts = defaultdict(Counter)
coord_counts = Counter()
skills = []


def read_limited(path: Path) -> str:
    try:
        if path.stat().st_size > MAX_BYTES:
            return ""
        if path.suffix.lower() == ".ipynb":
            nb = json.loads(path.read_text(errors="ignore"))
            parts = []
            for cell in nb.get("cells", []):
                src = cell.get("source", "")
                if isinstance(src, list):
                    src = "".join(src)
                parts.append(str(src))
            return "\n".join(parts)
        return path.read_text(errors="ignore")
    except Exception:
        return ""


def detect_game_id(text: str, path: Path) -> str:
    joined = f"{path.as_posix()}\n{text[:5000]}".lower()
    m = re.search(r"\b([a-z]{2}\d{2})[-_][0-9a-f]{6,}\b", joined)
    if m:
        return m.group(1)
    m = re.search(r"\b(game_id|game)\s*[:=]\s*['\"]?([a-z]{2}\d{2})", joined)
    if m:
        return m.group(2)
    return "global"


def register_source_skill(path: Path, text: str, source_kind: str):
    lower = text.lower()
    scores = {}
    for comp, pats in COMPONENT_PATTERNS.items():
        s = 0
        for pat in pats:
            s += lower.count(pat.lower())
        if s:
            scores[comp] = s
    if not scores:
        return
    actions = ACTION_REGEX.findall(text)
    gid = detect_game_id(text, path)
    for a in actions:
        action_counts[a] += 1
        by_game_action_counts[gid][a] += 1
    for x, y in COORD_REGEX.findall(text):
        try:
            xi, yi = int(x), int(y)
            if 0 <= xi < 64 and 0 <= yi < 64:
                coord_counts[(xi, yi)] += 1
        except Exception:
            pass
    value = 0.0
    weights = {
        "agent_contract": 2.0,
        "action_api": 2.0,
        "click_targeting": 1.6,
        "object_extraction": 1.8,
        "state_graph": 2.4,
        "replay_memory": 1.4,
        "world_model": 2.6,
        "stagnation_recovery": 1.3,
        "training_loop": 1.1,
        "skill_library": 1.2,
    }
    for k, s in scores.items():
        value += weights.get(k, 1.0) * min(12, s)
    sid = hashlib.blake2b(str(path).encode(), digest_size=6).hexdigest()
    top_components = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:6]
    skills.append({
        "skill_id": f"src_{sid}",
        "name": f"{source_kind}:{path.name}",
        "description": f"Mined static solver components from {path.as_posix()}",
        "trigger_conditions": [k for k, _ in top_components] + [gid],
        "components": dict(top_components),
        "source_path": path.as_posix(),
        "confidence": round(min(0.98, 0.35 + value / 80.0), 4),
        "success_rate": round(min(0.90, 0.25 + value / 110.0), 4),
        "examples_count": max(1, len(actions)),
        "action_template": ",".join(sorted(set(actions))) if actions else "structural_explore",
        "solver_value": round(value, 4),
    })
    component_hits.append({"path": path.as_posix(), "kind": source_kind, "game": gid, "scores": scores, "value": round(value, 4)})


def ingest_skill_json(path: Path, data: Any):
    loaded = []
    if isinstance(data, dict):
        if "skills" in data and isinstance(data["skills"], list):
            loaded = data["skills"]
        elif all(isinstance(v, dict) for v in data.values()):
            loaded = list(data.values())
    elif isinstance(data, list):
        loaded = [x for x in data if isinstance(x, dict)]
    for i, item in enumerate(loaded[:500]):
        text = json.dumps(item, sort_keys=True)[:5000]
        sid = item.get("skill_id") or f"json_{hashlib.blake2b((str(path)+str(i)).encode(), digest_size=6).hexdigest()}"
        skills.append({
            "skill_id": str(sid),
            "name": str(item.get("name", f"json_skill:{path.name}")),
            "description": str(item.get("description", f"Loaded compatible skill object from {path.as_posix()}")),
            "trigger_conditions": list(item.get("trigger_conditions", []))[:20] if isinstance(item.get("trigger_conditions", []), list) else [],
            "components": {"existing_skill_library": 1},
            "source_path": path.as_posix(),
            "confidence": float(item.get("confidence", 0.55)) if str(item.get("confidence", "")).replace(".", "", 1).isdigit() else 0.55,
            "success_rate": float(item.get("success_rate", 0.45)) if str(item.get("success_rate", "")).replace(".", "", 1).isdigit() else 0.45,
            "examples_count": int(item.get("examples_count", 1)) if str(item.get("examples_count", "1")).isdigit() else 1,
            "action_template": str(item.get("action_template", "loaded_skill")),
            "solver_value": 35.0,
        })
        for a in ACTION_REGEX.findall(text):
            action_counts[a] += 2


def preview_dataset(path: Path):
    suffix = path.suffix.lower()
    rows_seen = 0
    keys = Counter()
    try:
        if suffix == ".jsonl":
            with path.open("r", errors="ignore") as f:
                for line in f:
                    if rows_seen >= 2000:
                        break
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except Exception:
                        continue
                    rows_seen += 1
                    if isinstance(obj, dict):
                        keys.update(obj.keys())
                        text = json.dumps(obj)[:3000]
                        gid = str(obj.get("game_id") or obj.get("game") or detect_game_id(text, path)).split("-")[0].lower()
                        for a in ACTION_REGEX.findall(text):
                            action_counts[a] += 3
                            by_game_action_counts[gid][a] += 3
                        for x, y in COORD_REGEX.findall(text):
                            xi, yi = int(x), int(y)
                            if 0 <= xi < 64 and 0 <= yi < 64:
                                coord_counts[(xi, yi)] += 2
        elif suffix in {".csv", ".tsv"}:
            delim = "\t" if suffix == ".tsv" else ","
            with path.open("r", errors="ignore", newline="") as f:
                reader = csv.DictReader(f, delimiter=delim)
                for row in reader:
                    if rows_seen >= 2000:
                        break
                    rows_seen += 1
                    keys.update(row.keys())
                    text = json.dumps(row)[:3000]
                    gid = str(row.get("game_id") or row.get("game") or detect_game_id(text, path)).split("-")[0].lower()
                    for a in ACTION_REGEX.findall(text):
                        action_counts[a] += 2
                        by_game_action_counts[gid][a] += 2
                    for x, y in COORD_REGEX.findall(text):
                        xi, yi = int(x), int(y)
                        if 0 <= xi < 64 and 0 <= yi < 64:
                            coord_counts[(xi, yi)] += 1
        elif suffix == ".json":
            text = read_limited(path)
            if text:
                try:
                    data = json.loads(text)
                    if "skill" in path.name.lower() or "skills" in text[:2000].lower():
                        ingest_skill_json(path, data)
                    # Also mine generic JSON text for action/coor priors.
                    register_source_skill(path, text, "json")
                    rows_seen = 1
                except Exception:
                    pass
        elif suffix == ".parquet":
            try:
                import pandas as pd
                df = pd.read_parquet(path)
                sample = df.head(2000)
                rows_seen = len(sample)
                keys.update(map(str, sample.columns))
                for _, row in sample.iterrows():
                    d = {str(k): row[k] for k in sample.columns}
                    text = json.dumps(d, default=str)[:3000]
                    gid = str(d.get("game_id") or d.get("game") or detect_game_id(text, path)).split("-")[0].lower()
                    for a in ACTION_REGEX.findall(text):
                        action_counts[a] += 2
                        by_game_action_counts[gid][a] += 2
                    for x, y in COORD_REGEX.findall(text):
                        xi, yi = int(x), int(y)
                        if 0 <= xi < 64 and 0 <= yi < 64:
                            coord_counts[(xi, yi)] += 1
            except Exception:
                pass
    except Exception:
        pass
    return {"path": path.as_posix(), "rows_previewed": rows_seen, "keys": dict(keys.most_common(25))}

# Inventory normal files.
dataset_summaries = []
if KAGGLE_INPUT.exists():
    for path in KAGGLE_INPUT.rglob("*"):
        if not path.is_file():
            continue
        try:
            rel = path.relative_to(KAGGLE_INPUT).as_posix()
        except Exception:
            rel = path.as_posix()
        suffix = path.suffix.lower()
        size = path.stat().st_size
        inventory.append({"path": path.as_posix(), "rel": rel, "suffix": suffix, "size": size})
        if suffix in TEXT_EXTS and size <= MAX_BYTES:
            txt = read_limited(path)
            if txt:
                register_source_skill(path, txt, suffix.lstrip("."))
        if suffix in DATA_EXTS and size <= MAX_BYTES:
            dataset_summaries.append(preview_dataset(path))
        if suffix in ARCHIVE_EXTS and size <= MAX_BYTES:
            try:
                with zipfile.ZipFile(path) as zf:
                    names = zf.namelist()[:2000]
                    archive_text = "\n".join(names)
                    register_source_skill(path, archive_text, "archive_index")
                    inventory.append({"path": path.as_posix(), "archive_members_preview": names[:80], "member_count_previewed": len(names)})
            except Exception:
                pass

# Normalize priors to small floats.
def norm_counter(c: Counter) -> Dict[str, float]:
    if not c:
        return {}
    m = max(c.values()) or 1
    return {k: round(v / m, 4) for k, v in c.most_common()}

action_priors = {"global": norm_counter(action_counts), "by_game": {g: norm_counter(c) for g, c in by_game_action_counts.items() if c}}
coordinate_priors = [
    {"x": x, "y": y, "weight": round(v / max(1, coord_counts.most_common(1)[0][1]), 4)}
    for (x, y), v in coord_counts.most_common(128)
]

# Add core fallback skills even if inputs are sparse.
core_skills = [
    {
        "skill_id": "core_graph_explore",
        "name": "Graph-state exploration and no-change avoidance",
        "description": "Track state hashes, avoid repeated ineffective actions, and seek untested action/state transitions.",
        "trigger_conditions": ["state_graph", "visited", "stagnation_recovery"],
        "components": {"state_graph": 3, "stagnation_recovery": 2},
        "source_path": "built_in_core",
        "confidence": 0.72,
        "success_rate": 0.48,
        "examples_count": 1,
        "action_template": "state_hash->untried_action",
        "solver_value": 50.0,
    },
    {
        "skill_id": "core_object_click",
        "name": "Object-component coordinate targeting",
        "description": "Segment non-zero connected components and click centroids, edges, corners, and high-salience probes.",
        "trigger_conditions": ["object_extraction", "click_targeting", "ACTION6"],
        "components": {"object_extraction": 3, "click_targeting": 3},
        "source_path": "built_in_core",
        "confidence": 0.70,
        "success_rate": 0.45,
        "examples_count": 1,
        "action_template": "ACTION6(x,y)",
        "solver_value": 48.0,
    },
]
skills.extend(core_skills)

# Rank and dedupe skills.
seen = set()
ranked = []
for s in sorted(skills, key=lambda x: (float(x.get("solver_value", 0)), float(x.get("confidence", 0))), reverse=True):
    sid = str(s.get("skill_id"))
    if sid in seen:
        continue
    seen.add(sid)
    ranked.append(s)

skill_pack = {
    "schema": "nine.arc.component_skill.v3.competition_scored",
    "created_at": time.time(),
    "input_root": str(KAGGLE_INPUT),
    "skills": ranked[:512],
    "action_priors": action_priors,
    "coordinate_priors": coordinate_priors,
    "component_rankings": sorted(component_hits, key=lambda x: x.get("value", 0), reverse=True)[:1000],
}

(SKILL_DIR / "skill_library.json").write_text(json.dumps(skill_pack, indent=2))
(REPORT_DIR / "input_inventory.json").write_text(json.dumps(inventory[:5000], indent=2))
(REPORT_DIR / "component_rankings.json").write_text(json.dumps(skill_pack["component_rankings"], indent=2))
(REPORT_DIR / "dataset_summaries.json").write_text(json.dumps(dataset_summaries[:1000], indent=2))
(REPORT_DIR / "transition_priors.json").write_text(json.dumps({"action_priors": action_priors, "coordinate_priors": coordinate_priors}, indent=2))

print("files inventoried:", len(inventory))
print("skills ranked:", len(ranked))
print("global action priors:", action_priors.get("global", {}))
print("top components:")
for h in skill_pack["component_rankings"][:10]:
    print(" ", h.get("value"), h.get("kind"), h.get("path"))


In [ ]:

# Write the exact official-framework agent file.
# /tmp/my_agent.py is what the ARC-AGI-3-Agents framework consumes.
# /kaggle/working/agent/my_agent.py is intentionally also emitted for audit/download.
from pathlib import Path
AGENT_CODE = '"""\nARC-AGI-3 Component-Skill Agent v3\n\nOfficial framework contract:\n- File is copied to ARC-AGI-3-Agents/agents/templates/my_agent.py\n- Class name must be MyAgent\n- Subclass agents.agent.Agent\n- choose_action returns a GameAction, with complex action data attached by action.set_data(...)\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import Counter, defaultdict, deque\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Tuple\n\nfrom arcengine import FrameData, GameAction, GameState\nfrom agents.agent import Agent\n\n\nGRID_SIZE = 64\n\n\ndef _safe_name(action: Any) -> str:\n    return getattr(action, "name", str(action)).upper()\n\n\ndef _game_prefix(game_id: str) -> str:\n    return (game_id or "unknown").split("-")[0].lower()\n\n\ndef _grid_from_frame(frame_obj: Any) -> List[List[int]]:\n    """Normalize FrameData/FrameDataRaw into a 2-D integer grid."""\n    frame = getattr(frame_obj, "frame", None)\n    if frame is None:\n        frame = getattr(frame_obj, "grid", None)\n    if frame is None:\n        return []\n\n    # Convert numpy-like arrays without requiring numpy.\n    if hasattr(frame, "tolist"):\n        frame = frame.tolist()\n\n    if not frame:\n        return []\n\n    # Raw API shape may be [frames][rows][cols]. Use latest visual frame.\n    try:\n        if isinstance(frame, list) and frame and isinstance(frame[0], list) and frame[0] and isinstance(frame[0][0], list):\n            frame = frame[-1]\n    except Exception:\n        return []\n\n    # Force rectangular-ish 2-D list of ints.\n    out: List[List[int]] = []\n    try:\n        for row in frame:\n            if hasattr(row, "tolist"):\n                row = row.tolist()\n            out.append([int(x) for x in row])\n    except Exception:\n        return []\n    return out\n\n\ndef _hash_grid(grid: List[List[int]]) -> str:\n    if not grid:\n        return "empty"\n    # Use a compact deterministic hash; enough for state graph keys.\n    return hashlib.blake2b(str(grid).encode("utf-8", "ignore"), digest_size=8).hexdigest()\n\n\ndef _grid_changed(a: List[List[int]], b: List[List[int]]) -> bool:\n    return bool(a and b and a != b)\n\n\ndef _iter_nonzero(grid: List[List[int]]) -> Iterable[Tuple[int, int, int]]:\n    for y, row in enumerate(grid):\n        for x, val in enumerate(row):\n            if val:\n                yield x, y, val\n\n\ndef _extract_objects(grid: List[List[int]], max_objects: int = 512) -> List[Dict[str, Any]]:\n    if not grid or not grid[0]:\n        return []\n    h, w = len(grid), len(grid[0])\n    seen = [[False] * w for _ in range(h)]\n    objects: List[Dict[str, Any]] = []\n\n    for y in range(h):\n        for x in range(w):\n            color = grid[y][x]\n            if color == 0 or seen[y][x]:\n                continue\n            q = deque([(x, y)])\n            seen[y][x] = True\n            pixels: List[Tuple[int, int]] = []\n            while q:\n                cx, cy = q.popleft()\n                pixels.append((cx, cy))\n                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):\n                    nx, ny = cx + dx, cy + dy\n                    if 0 <= nx < w and 0 <= ny < h and not seen[ny][nx] and grid[ny][nx] == color:\n                        seen[ny][nx] = True\n                        q.append((nx, ny))\n            xs = [p[0] for p in pixels]\n            ys = [p[1] for p in pixels]\n            min_x, max_x = min(xs), max(xs)\n            min_y, max_y = min(ys), max(ys)\n            objects.append(\n                {\n                    "color": int(color),\n                    "pixels": pixels,\n                    "area": len(pixels),\n                    "bbox": (min_x, min_y, max_x, max_y),\n                    "cx": int(round(sum(xs) / len(xs))),\n                    "cy": int(round(sum(ys) / len(ys))),\n                    "width": max_x - min_x + 1,\n                    "height": max_y - min_y + 1,\n                }\n            )\n            if len(objects) >= max_objects:\n                return objects\n    return objects\n\n\ndef _load_skill_pack() -> Dict[str, Any]:\n    candidates = [\n        Path("/kaggle/working/data/skills/skill_library.json"),\n        Path("/tmp/skill_library.json"),\n        Path("data/skills/skill_library.json"),\n    ]\n    for p in candidates:\n        try:\n            if p.exists():\n                data = json.loads(p.read_text())\n                if isinstance(data, dict):\n                    return data\n        except Exception:\n            pass\n    return {"schema": "empty", "skills": [], "action_priors": {}, "coordinate_priors": []}\n\n\ndef _available_actions_from_context(agent: Any, latest_frame: Any) -> List[GameAction]:\n    """Prefer framework env.action_space; fall back to obs available_actions; then all enum actions."""\n    actions: List[Any] = []\n    env = getattr(agent, "arc_env", None)\n    if env is not None:\n        try:\n            raw = getattr(env, "action_space", None)\n            if raw:\n                actions = list(raw)\n        except Exception:\n            actions = []\n\n    if not actions:\n        raw = getattr(latest_frame, "available_actions", None)\n        if raw:\n            actions = list(raw)\n\n    converted: List[GameAction] = []\n    for a in actions:\n        if isinstance(a, GameAction):\n            converted.append(a)\n        elif isinstance(a, int):\n            if a == 0 and hasattr(GameAction, "RESET"):\n                converted.append(GameAction.RESET)\n            elif hasattr(GameAction, f"ACTION{a}"):\n                converted.append(getattr(GameAction, f"ACTION{a}"))\n        elif isinstance(a, str):\n            name = a.upper().split(".")[-1]\n            if hasattr(GameAction, name):\n                converted.append(getattr(GameAction, name))\n\n    if not converted:\n        try:\n            converted = [a for a in GameAction]\n        except Exception:\n            converted = []\n\n    # De-duplicate while preserving enum identity.\n    out: List[GameAction] = []\n    for a in converted:\n        if a not in out:\n            out.append(a)\n    return out\n\n\nclass MyAgent(Agent):\n    """Verifier-driven structural explorer seeded by mined Kaggle input skills."""\n\n    MAX_ACTIONS = int(os.environ.get("NINE_ARC_MAX_ACTIONS", "240"))\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.skill_pack = _load_skill_pack()\n        self.game_prefix = _game_prefix(getattr(self, "game_id", "unknown"))\n        seed_base = 918 + sum(ord(c) for c in self.game_prefix) + int(time.time() * 1000) % 997\n        random.seed(seed_base)\n\n        self.last_grid: List[List[int]] = []\n        self.last_hash = ""\n        self.last_action: Optional[GameAction] = None\n        self.last_action_name = ""\n        self.last_data: Dict[str, Any] = {}\n        self.same_state_steps = 0\n        self.action_counter_local = 0\n        self.state_visits: Counter[str] = Counter()\n        self.transition_graph: Dict[str, Dict[str, str]] = defaultdict(dict)\n        self.bad_actions: Counter[str] = Counter()\n        self.good_actions: Counter[str] = Counter()\n        self.click_queue: deque[Tuple[int, int, str]] = deque()\n        self.tried_clicks_by_state: Dict[str, set] = defaultdict(set)\n        self.click_strategy_index = 0\n        self.simple_cycle_index = 0\n\n        priors = self.skill_pack.get("action_priors", {}) if isinstance(self.skill_pack, dict) else {}\n        self.global_action_priors = dict(priors.get("global", {})) if isinstance(priors, dict) else {}\n        self.game_action_priors = dict(priors.get("by_game", {}).get(self.game_prefix, {})) if isinstance(priors, dict) else {}\n        self.coord_priors = list(self.skill_pack.get("coordinate_priors", [])) if isinstance(self.skill_pack, dict) else []\n\n    @property\n    def name(self) -> str:\n        return f"{super().name}.component_skill_v3.{self.MAX_ACTIONS}"\n\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        if latest_frame.state is GameState.WIN:\n            return True\n        if self.action_counter_local >= self.MAX_ACTIONS:\n            return True\n        return False\n\n    def _score_action(self, action: GameAction, state_hash: str) -> float:\n        name = _safe_name(action)\n        score = 1.0\n        score += 2.5 * float(self.game_action_priors.get(name, 0.0))\n        score += 1.25 * float(self.global_action_priors.get(name, 0.0))\n        score += 0.35 * self.good_actions[name]\n        score -= 0.55 * self.bad_actions[name]\n        if state_hash and self.transition_graph.get(state_hash, {}).get(name):\n            score -= 0.4\n        if action is GameAction.RESET:\n            score -= 4.0\n        try:\n            if action.is_complex():\n                score += 0.2\n        except Exception:\n            pass\n        return score + random.random() * 0.05\n\n    def _update_feedback(self, grid: List[List[int]]) -> None:\n        state_hash = _hash_grid(grid)\n        self.state_visits[state_hash] += 1\n\n        if self.last_action is not None and self.last_hash:\n            name = self.last_action_name\n            self.transition_graph[self.last_hash][name] = state_hash\n            if _grid_changed(self.last_grid, grid):\n                self.good_actions[name] += 1\n                self.same_state_steps = 0\n            else:\n                self.bad_actions[name] += 1\n                self.same_state_steps += 1\n                if name == "ACTION6":\n                    self.click_strategy_index += 1\n        self.last_hash = state_hash\n\n    def _seed_click_queue(self, grid: List[List[int]], state_hash: str) -> None:\n        if self.click_queue:\n            return\n        if not grid or not grid[0]:\n            self.click_queue.append((GRID_SIZE // 2, GRID_SIZE // 2, "empty_center"))\n            return\n\n        h, w = len(grid), len(grid[0])\n        candidates: List[Tuple[float, int, int, str]] = []\n        objects = _extract_objects(grid)\n\n        # Object-centric clicks: largest, smallest, corners, colored centroids.\n        if objects:\n            sorted_objs = sorted(objects, key=lambda o: o["area"], reverse=True)\n            for rank, obj in enumerate(sorted_objs[:40]):\n                min_x, min_y, max_x, max_y = obj["bbox"]\n                size_bonus = math.log1p(obj["area"])\n                rarity_bonus = 1.0 / max(1, sum(1 for o in objects if o["color"] == obj["color"]))\n                base = size_bonus + rarity_bonus - rank * 0.03\n                points = [\n                    (obj["cx"], obj["cy"], "object_centroid"),\n                    (min_x, min_y, "object_topleft"),\n                    (max_x, max_y, "object_bottomright"),\n                    ((min_x + max_x) // 2, min_y, "object_topmid"),\n                    ((min_x + max_x) // 2, max_y, "object_bottommid"),\n                    (min_x, (min_y + max_y) // 2, "object_leftmid"),\n                    (max_x, (min_y + max_y) // 2, "object_rightmid"),\n                ]\n                for x, y, reason in points:\n                    candidates.append((base, x, y, reason))\n\n        # Dataset coordinate priors mined from loaded replay notebooks/datasets.\n        for item in self.coord_priors[:64]:\n            try:\n                x = int(item.get("x", 32))\n                y = int(item.get("y", 32))\n                weight = float(item.get("weight", 1.0))\n                if 0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE:\n                    candidates.append((2.0 + weight, x, y, "dataset_coord_prior"))\n            except Exception:\n                continue\n\n        # General grid probes: center, quadrants, borders.\n        probes = [\n            (w // 2, h // 2, "center"),\n            (w // 4, h // 4, "q1"),\n            ((3 * w) // 4, h // 4, "q2"),\n            (w // 4, (3 * h) // 4, "q3"),\n            ((3 * w) // 4, (3 * h) // 4, "q4"),\n            (1, 1, "corner_tl"),\n            (w - 2, 1, "corner_tr"),\n            (1, h - 2, "corner_bl"),\n            (w - 2, h - 2, "corner_br"),\n        ]\n        for x, y, reason in probes:\n            candidates.append((0.8, max(0, min(GRID_SIZE - 1, x)), max(0, min(GRID_SIZE - 1, y)), reason))\n\n        candidates.sort(reverse=True)\n        tried = self.tried_clicks_by_state[state_hash]\n        for _, x, y, reason in candidates:\n            x = int(max(0, min(GRID_SIZE - 1, x)))\n            y = int(max(0, min(GRID_SIZE - 1, y)))\n            key = (x, y)\n            if key not in tried:\n                self.click_queue.append((x, y, reason))\n                tried.add(key)\n            if len(self.click_queue) >= 32:\n                break\n        if not self.click_queue:\n            self.click_queue.append((random.randrange(GRID_SIZE), random.randrange(GRID_SIZE), "random_unseen"))\n\n    def _choose_simple_action(self, actions: List[GameAction], state_hash: str) -> GameAction:\n        legal = [a for a in actions if a is not GameAction.RESET]\n        if not legal:\n            return GameAction.RESET\n        scored = sorted(((self._score_action(a, state_hash), a) for a in legal), key=lambda p: p[0], reverse=True)\n        # If stagnant, force exploration among less-used actions.\n        if self.same_state_steps >= 3 and len(scored) > 1:\n            idx = self.simple_cycle_index % min(len(scored), 5)\n            self.simple_cycle_index += 1\n            return scored[idx][1]\n        return scored[0][1]\n\n    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:\n        self.action_counter_local += 1\n\n        # Official reset behavior: first observation or death/game-over.\n        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\n            self.last_action = GameAction.RESET\n            self.last_action_name = "RESET"\n            self.last_data = {}\n            GameAction.RESET.reasoning = {"why": "reset from not-played/game-over state"}\n            return GameAction.RESET\n\n        grid = _grid_from_frame(latest_frame)\n        self._update_feedback(grid)\n        state_hash = _hash_grid(grid)\n        actions = _available_actions_from_context(self, latest_frame)\n        legal_non_reset = [a for a in actions if a is not GameAction.RESET]\n        if not legal_non_reset:\n            self.last_action = GameAction.RESET\n            self.last_action_name = "RESET"\n            self.last_data = {}\n            return GameAction.RESET\n\n        complex_actions = []\n        for a in legal_non_reset:\n            try:\n                if a.is_complex():\n                    complex_actions.append(a)\n            except Exception:\n                pass\n\n        use_click = False\n        if complex_actions:\n            # Prefer clicks early and when simple actions are failing; mix in simple actions to discover controls.\n            use_click = self.same_state_steps >= 2 or self.action_counter_local % 3 != 0 or not self.good_actions\n\n        if use_click:\n            self._seed_click_queue(grid, state_hash)\n            x, y, reason = self.click_queue.popleft()\n            action = sorted(complex_actions, key=lambda a: self._score_action(a, state_hash), reverse=True)[0]\n            data = {"x": int(x), "y": int(y)}\n            try:\n                action.set_data(data)\n            except Exception:\n                pass\n            action.reasoning = {\n                "why": "component_skill_v3 structural click",\n                "game": self.game_prefix,\n                "state": state_hash,\n                "target": data,\n                "reason": reason,\n                "same_state_steps": self.same_state_steps,\n            }\n            self.last_action = action\n            self.last_action_name = _safe_name(action)\n            self.last_data = data\n            self.last_grid = grid\n            return action\n\n        action = self._choose_simple_action(legal_non_reset, state_hash)\n        try:\n            if action.is_complex():\n                self._seed_click_queue(grid, state_hash)\n                x, y, reason = self.click_queue.popleft()\n                data = {"x": int(x), "y": int(y)}\n                action.set_data(data)\n                action.reasoning = {"why": "complex fallback", "target": data, "reason": reason}\n                self.last_data = data\n            else:\n                action.reasoning = {\n                    "why": "component_skill_v3 simple action",\n                    "game": self.game_prefix,\n                    "state": state_hash,\n                    "same_state_steps": self.same_state_steps,\n                    "score": self._score_action(action, state_hash),\n                }\n                self.last_data = {}\n        except Exception:\n            pass\n\n        self.last_action = action\n        self.last_action_name = _safe_name(action)\n        self.last_grid = grid\n        return action\n'
Path('/tmp/my_agent.py').write_text(AGENT_CODE)
(AGENT_DIR / 'my_agent.py').write_text(AGENT_CODE)
print('/tmp/my_agent.py bytes:', Path('/tmp/my_agent.py').stat().st_size)
print('/kaggle/working/agent/my_agent.py exists:', (AGENT_DIR / 'my_agent.py').exists())


In [ ]:

# Offline syntax smoke check; does not require importing arc_agi/arcengine.
import py_compile, json, os
try:
    py_compile.compile(str(AGENT_DIR / "my_agent.py"), doraise=True)
    smoke = {"my_agent_py_compile": True}
except Exception as e:
    smoke = {"my_agent_py_compile": False, "error": repr(e)}
(REPORT_DIR / "offline_syntax_smoke.json").write_text(json.dumps(smoke, indent=2))
print(json.dumps(smoke, indent=2))


In [ ]:

# Official competition-scored execution path.
# During Kaggle Save & Run All: emit the required dummy submission.parquet so commit succeeds.
# During Kaggle Competition Rerun: run the official gateway/framework path; gateway emits the real submission.parquet.
import os, json, subprocess, sys, time, shutil
from pathlib import Path

run_report = {
    "competition_rerun": bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN")),
    "submission_path": str(WORK_DIR / "submission.parquet"),
    "agent_output_path": str(AGENT_DIR / "my_agent.py"),
    "framework_copied": False,
    "gateway_checked": False,
    "main_returncode": None,
    "errors": [],
}

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    # Wait for ARC gateway sidecar exactly like official starter.
    try:
        cmd = ["curl", "--fail", "--retry", "999", "--retry-all-errors", "--retry-delay", "5", "--retry-max-time", "600", "http://gateway:8001/api/games"]
        subprocess.run(cmd, check=True)
        run_report["gateway_checked"] = True
    except Exception as e:
        run_report["errors"].append(f"gateway wait/check failed: {e!r}")
        raise

    src_framework = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    dst_framework = WORK_DIR / "ARC-AGI-3-Agents"
    if dst_framework.exists():
        shutil.rmtree(dst_framework)
    shutil.copytree(src_framework, dst_framework)
    run_report["framework_copied"] = True

    dst_agent = dst_framework / "agents" / "templates" / "my_agent.py"
    shutil.copyfile("/tmp/my_agent.py", dst_agent)

    # Register MyAgent without importing optional heavy templates.
    init_text = """from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
"""
    (dst_framework / "agents" / "__init__.py").write_text(init_text)

    # Point framework at gateway sidecar. Gateway generates the real submission.parquet.
    env_text = """SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
"""
    (dst_framework / ".env").write_text(env_text)

    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"
    env["NINE_ARC_MAX_ACTIONS"] = str(CONFIG["max_actions"])
    proc = subprocess.run([sys.executable, "main.py", "--agent", "myagent"], cwd=str(dst_framework), env=env)
    run_report["main_returncode"] = proc.returncode
    if proc.returncode != 0:
        run_report["errors"].append(f"official framework main.py failed: returncode={proc.returncode}")
        (REPORT_DIR / "competition_run_report.json").write_text(json.dumps(run_report, indent=2))
        raise SystemExit(proc.returncode)

    if not (WORK_DIR / "submission.parquet").exists():
        run_report["errors"].append("gateway/framework completed but submission.parquet was not found")
        (REPORT_DIR / "competition_run_report.json").write_text(json.dumps(run_report, indent=2))
        raise FileNotFoundError("/kaggle/working/submission.parquet missing after competition rerun")

else:
    # Commit/Save-and-run-all mode: create the required placeholder output.
    # This is not the scored file. The scored file is generated by gateway in competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    submission.to_parquet(WORK_DIR / "submission.parquet", index=False)
    run_report["main_returncode"] = 0
    print(submission.head())

# Manifest of the two required files plus supporting artifacts.
outputs = {
    "required_files": {
        "submission.parquet": str(WORK_DIR / "submission.parquet"),
        "agent/my_agent.py": str(AGENT_DIR / "my_agent.py"),
    },
    "supporting_files": {
        "skill_library": str(SKILL_DIR / "skill_library.json"),
        "reports_dir": str(REPORT_DIR),
    },
    "exists": {
        "submission.parquet": (WORK_DIR / "submission.parquet").exists(),
        "agent/my_agent.py": (AGENT_DIR / "my_agent.py").exists(),
    },
}
run_report["outputs"] = outputs
(REPORT_DIR / "competition_run_report.json").write_text(json.dumps(run_report, indent=2))
(REPORT_DIR / "output_files_manifest.json").write_text(json.dumps(outputs, indent=2))
print(json.dumps(outputs, indent=2))
print("submission.parquet size:", (WORK_DIR / "submission.parquet").stat().st_size if (WORK_DIR / "submission.parquet").exists() else "missing")
print("agent/my_agent.py size:", (AGENT_DIR / "my_agent.py").stat().st_size if (AGENT_DIR / "my_agent.py").exists() else "missing")
